# Transformers, end to end

A pretrained model doing the thing transformers exist to do: turning the same word into different vectors depending on what surrounds it.

Nothing here is trained. `from_pretrained` downloads DistilBERT once (about 250 MB) and caches it; the rest is a single forward pass under `torch.no_grad()`, which is what inference looks like in practice. Loading prints a report about discarded pretraining-head weights — that is expected when you load a base model rather than a task head, not an error.

`attn_implementation="eager"` is needed to get attention weights back. The faster default kernels never materialise the matrix.

In [1]:
import torch
from transformers import AutoModel, AutoTokenizer

MODEL = "distilbert-base-uncased"

# 1. Downloaded once, then cached. eager attention so the weights come back.
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL, attn_implementation="eager")
model.eval()

# 2. The same word, two different sentences
sentences = ["the bank raised interest rates", "he sat on the river bank"]
batch = tokenizer(sentences, return_tensors="pt", padding=True)
tokens = [tokenizer.convert_ids_to_tokens(ids) for ids in batch["input_ids"]]
print("tokens[0]:", tokens[0])
print("tokens[1]:", tokens[1])

# 3. One forward pass. No gradients: this is inference, not training.
with torch.no_grad():
    out = model(**batch, output_attentions=True)

print("\nhidden states:", tuple(out.last_hidden_state.shape), "(batch, tokens, dim)")
print("layers:", len(out.attentions), " heads per layer:", out.attentions[0].shape[1])

# 4. Locate "bank" in each sentence
bank_id = tokenizer.convert_tokens_to_ids("bank")
where = [(ids == bank_id).nonzero()[0].item() for ids in batch["input_ids"]]

static = model.embeddings.word_embeddings(torch.tensor([bank_id]))[0]
contextual = [out.last_hidden_state[i, w] for i, w in enumerate(where)]
cos = torch.nn.functional.cosine_similarity

print(f"\nstatic embedding vs itself:  {cos(static, static, dim=0):.4f}")
print(f"contextual 'bank' vs 'bank': {cos(contextual[0], contextual[1], dim=0):.4f}")

# 5. What each "bank" looked at, averaged over heads, in two layers
for layer in (0, len(out.attentions) - 1):
    print()
    print(f"layer {layer}:")
    for i, sentence in enumerate(sentences):
        weights = out.attentions[layer][i, :, where[i], :].mean(0)
        top = torch.topk(weights, 3)
        pairs = [f"{tokens[i][j]}={v:.2f}" for v, j in zip(top.values, top.indices)]
        print(f"  '{sentence}' -> {', '.join(pairs)}")

tokens[0]: ['[CLS]', 'the', 'bank', 'raised', 'interest', 'rates', '[SEP]', '[PAD]']
tokens[1]: ['[CLS]', 'he', 'sat', 'on', 'the', 'river', 'bank', '[SEP]']

hidden states: (2, 8, 768) (batch, tokens, dim)
layers: 6  heads per layer: 12

static embedding vs itself:  1.0000
contextual 'bank' vs 'bank': 0.5595

layer 0:
  'the bank raised interest rates' -> [CLS]=0.25, the=0.14, raised=0.14
  'he sat on the river bank' -> river=0.35, [CLS]=0.19, [SEP]=0.17

layer 5:
  'the bank raised interest rates' -> [SEP]=0.72, bank=0.08, the=0.06
  'he sat on the river bank' -> [SEP]=0.71, bank=0.07, the=0.05

## What the output is telling you

- **`(2, 8, 768)`** is (sentences, tokens, dimensions). Both sentences are padded to 8 tokens so they fit one batch — the shorter one carries `[PAD]`. Six layers, twelve heads each.
- **The same word, two different vectors.** The static input embedding for `bank` is one fixed row of a lookup table: identical in both sentences, cosine 1.0000. After six layers of attention the two vectors have drifted to **cosine 0.5595**. That gap *is* the contextualisation, and it is the reason these models replaced word2vec.
- **Layer 0 shows the mechanism working.** In "he sat on the river bank", `bank` puts 0.35 of its attention on `river` — the token that disambiguates it. In the interest-rate sentence there is no such neighbour and the weight scatters.
- **Layer 5 shows why attention maps are not explanations.** By the last layer both sentences dump ~0.72 onto `[SEP]`. Heads with nothing useful to do park their weight on a structural token; this is a well-documented no-op, and it means a pretty attention heat-map is not evidence of reasoning.

## When to reach for this

Use a pretrained transformer when meaning depends on context: classification where phrasing matters, retrieval where "bank" must not match both senses, extraction, or anything on raw text. Fine-tune it when you have labels; use the embeddings directly when you do not.

Do not reach for it on tabular data. On the breast-cancer dataset in the other notebooks, gradient boosting and even logistic regression beat it — transformers buy you the ability to learn a representation, which is wasted where the columns are already meaningful.

## Extend this notebook

- Swap in sentences of your own and watch the cosine move. Near-identical contexts should push it back toward 1.0.
- Pool `last_hidden_state` with the attention mask (mean over real tokens only) to get a sentence embedding, then compare sentences rather than words.
- Compare all six layers. Middle layers usually carry the most transferable features; the last is specialised toward the pretraining objective.
- For a task rather than an inspection, `pipeline("sentiment-analysis")` is three lines and downloads its own fine-tuned head.